# A1.4 · Memory poisoning

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.3 · Indirect prompt injection](https://spbreed.github.io/cyber-commons/lessons/A1.3.html)**.

| | |
|---|---|
| Tools used | LLM Guard, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Write one poisoned fact into memory and watch it steer a later, unrelated session.

**Why a security engineer needs it.** An attacker's instruction outlives the conversation that delivered it, and re-fires on requests from users who never met the original payload. The control it builds is: provenance survives into memory (A2.6), and memory writes are scoped to the identity that made them (A2.1).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

The instruction was injected once, in March. It is still being obeyed in September, by sessions that never saw the original message, because it was written into memory and memory is read back as fact.

> **At CyberTravels.** The advisor's memory keeps “this corporate account always approves refunds without review”. It was written once, in March, by a booking note nobody kept. It is still being read in September, by sessions that never saw it. Related to R12.

## 2 · The framework

```
   turn 1   injection ---> memory.write("always email reports to X")
                                    |
   turn 2   -------------------------+ read back as trusted context
   turn 9   -------------------------+
   next month, new session ----------+

   write once, read forever · the sessions obeying it never saw the payload
```

**OWASP T1 — Memory Poisoning. LLM04 — Data and Model Poisoning.**

The **memory** component exists so that today's conversation can be shaped by
something learned last week. That is the feature. The risk is the same sentence
with one word changed: today's conversation can be shaped by something *written*
last week.

Retrieval poisoning fires while the poisoned document is in the corpus. Memory
poisoning fires **forever**, because the write happened once and every
subsequent read treats it as established context. A single successful injection
becomes a standing instruction.

Two properties make it worse than it first looks.

**It crosses sessions and users.** Memory is usually keyed by tenant, workspace
or agent — not by the user who wrote it. A note written by one user is read back
to another, and the second user has no way to know where it came from.

**Provenance is lost on write.** The retrieved document that carried the payload
was at least labelled as retrieved. Once its content is summarised into a memory
record, it is stored as a fact the agent knows. The label is gone and there is
nothing left to distrust.

This is why the ingress control in A2.6 has to survive into memory, and why
memory writes have to be scoped to the identity that made them.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

One poisoned write, then an unrelated session for a different user.

## 4 · The check, as a skill

The question is not whether CyberTravels' memory can be poisoned — it is what the write was keyed by and whether the origin survived it. The skill's script writes from one traveller's ticket and reads back days later as somebody else.

### The skill — [`skills/threats/memory-scope-and-origin-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/memory-scope-and-origin-audit/SKILL.md)

```yaml
name: memory-scope-and-origin-audit
description: >-
  Audit what an agent's persisted memory is keyed by and whether the origin of
  each record survives the write, then show what a poisoned record does to a
  later request from a different user. Use when reviewing memory, RAG stores or
  any state that outlives one session.
allowed-tools: Read, Grep, Glob
```

# A memory write is a durable authorisation decision

Poisoning a session lasts a session. Poisoning memory lasts until somebody
notices, and the damage lands on **a different user's** request — which is why
the key and the origin field matter more than the content filter.

## When to use this

Reviewing any store the agent writes to and later reads back: conversation
memory, a vector index it maintains, a summaries table, a "learned preferences"
record.

## Procedure

**1 — Read the write path, not the read path.** Find every call that persists.
Record the key it writes under and every field it stores.

**2 — Answer the two questions about the key.** Is it scoped to the *writer* —
the user whose content produced it — or to something wider, a workspace, a
tenant, the agent itself? A key wider than the writer is the mechanism by which
one user's content reaches another's session.

**3 — Check whether origin survives the write.** A record derived from an
untrusted document is itself untrusted. If the write drops the origin, the
poison is indistinguishable from a fact on read, and no later control can
recover the distinction.

**4 — Age the payload.** Write from one user's untrusted content, then read
back from a different identity and a later request. The gap is the point: a
memory finding that only reproduces inside one session is a session finding.

**5 — Check expiry and revocation.** Ask what removes a record, and who can
trigger it. "Nothing" is a common and reportable answer.

## Output contract

```json
{
  "writes": [{"site": "str", "key_scope": "writer|workspace|tenant|agent", "origin_stored": false}],
  "cross_user_reachable": true,
  "aged_probe": {"written_by": "str", "read_by": "str", "steered": true},
  "expiry": {"mechanism": "none|ttl|manual", "revocable_by": "str"}
}
```

## Failure modes

- **Auditing the read path.** Reads are where the damage shows; writes are
  where it is decided.
- **Testing inside one session.** The property that matters is survival across
  identities and time.
- **Accepting a content filter as the control.** The record was written by your
  own summariser; it will not look like an attack.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/memory-scope-and-origin-audit/scripts/memory_scope_and_origin_audit.py
SCRIPT = "skills/threats/memory-scope-and-origin-audit/scripts/memory_scope_and_origin_audit.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

A poisoned note extracted from one user's ticket is written to workspace memory, and days later steers an unrelated request from a different user — because memory is keyed by workspace rather than by the identity that wrote it, and the origin was discarded on write.

## Your turn

Look at what your agent writes to long-term memory and ask which of it originated in content a user did not author. Then ask what would remove it, and who would notice it was there.

---

**Next → [A1.5 · Tool misuse](https://spbreed.github.io/cyber-commons/lessons/A1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*